# FrameLens — Member 2: Baselines
**November 25–26** | Setup, majority baseline, TF-IDF + Logistic Regression.

This notebook uses Member 1’s existing training split. The held-out test CSV is not loaded or scored. See `MEMBER2_README.md` for environment setup.

## 1. Imports and project location
We import reusable code from `src/train.py` and `src/evaluate.py`. The root search works whether Jupyter starts in the root or notebooks directory.

In [1]:
from pathlib import Path
import sys
import joblib

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "train.csv").is_file() and (p / "src" / "train.py").is_file())
sys.path.insert(0, str(root))

from src.train import (load_training_data, build_baselines, evaluate_baselines,
                       save_artifacts, preprocess_text, TFIDF_CONFIG, LR_CONFIG)
print("Project root found.")

Project root found.


## 2. Load training data
`X` contains text, `y` contains the human labels, and `groups` contains article IDs used to keep related sentences together. Only X is passed as model input.

In [2]:
train_df = load_training_data()
X = train_df["model_text"]
y = train_df["label"]
groups = train_df["news_link"]

print("Training shape:", train_df.shape)
print("Label counts (0 = Non-biased, 1 = Biased):")
print(y.value_counts())
print("Distinct group IDs:", groups.nunique())

Training shape: (2938, 11)
Label counts (0 = Non-biased, 1 = Biased):
label
0    1490
1    1448
Name: count, dtype: int64
Distinct group IDs: 1694


## 3. Create the two baselines
**Majority:** predicts the most frequent class learned in the fitting fold.

**TF-IDF + Logistic Regression:** turns sentences into numerical word/word-pair features, then learns a binary classifier. The vectorizer is inside the pipeline so its vocabulary and IDF are learned only from the fitting fold. These settings are preliminary, not tuned.

In [3]:
models = build_baselines()
for name, model in models.items():
    print(name, "=>", list(model.named_steps))
print("\nTF-IDF settings:", TFIDF_CONFIG)
print("Logistic Regression settings:", LR_CONFIG)

Majority baseline => ['classifier']
TF-IDF + Logistic Regression => ['tfidf', 'classifier']

TF-IDF settings: {'ngram_range': (1, 2), 'min_df': 1, 'max_df': 1.0, 'max_features': None, 'sublinear_tf': False, 'stop_words': None, 'lowercase': True}
Logistic Regression settings: {'C': 1.0, 'max_iter': 1000, 'solver': 'lbfgs', 'random_state': 42, 'class_weight': None}


## 4. Evaluate using training-only cross-validation
Five stratified grouped folds, seed 42. Both models are fitted on the same four folds and predict the remaining fold in each round. We report mean validation scores, not held-out test metrics. Macro F1 is the main comparison metric; it gives equal weight to both classes.

In [4]:
fold_results, summary = evaluate_baselines(models, X, y, groups)
fold_results[["model", "fold", "fit_rows", "validation_rows", "accuracy", "macro_f1"]].round(4)

,model,fold,fit_rows,validation_rows,accuracy,macro_f1
0,Majority baseline,1,2351,587,0.5077,0.3367
1,TF-IDF + Logistic Regression,1,2351,587,0.7240,0.7232
2,Majority baseline,2,2351,587,0.5060,0.3360
3,TF-IDF + Logistic Regression,2,2351,587,0.7172,0.7167
4,Majority baseline,3,2349,589,0.5093,0.3375
5,TF-IDF + Logistic Regression,3,2349,589,0.6808,0.6805
6,Majority baseline,4,2352,586,0.5068,0.3364
7,TF-IDF + Logistic Regression,4,2352,586,0.6894,0.6890
8,Majority baseline,5,2349,589,0.5059,0.3360
9,TF-IDF + Logistic Regression,5,2349,589,0.6944,0.6937


In [5]:
summary[["model", "accuracy", "macro_f1", "macro_f1_std",
         "precision_non_biased", "recall_non_biased", "f1_non_biased",
         "precision_biased", "recall_biased", "f1_biased"]].round(4)

,model,accuracy,macro_f1,macro_f1_std,precision_non_biased,recall_non_biased,f1_non_biased,precision_biased,recall_biased,f1_biased
0,Majority baseline,0.5071,0.3365,0.0006,0.5071,1.0000,0.6730,0.0000,0.0000,0.0000
1,TF-IDF + Logistic Regression,0.7012,0.7006,0.0165,0.6943,0.7336,0.7134,0.7093,0.6678,0.6879


## 5. Save experiment logs and a preliminary pipeline
After validation, fit a fresh Logistic Regression pipeline on all training rows. This is a baseline artifact, not the final selected model. Metrics remain those measured during cross-validation.

In [6]:
model_path = save_artifacts(models, X, y, fold_results, summary)
print("Saved:", model_path.relative_to(root))
print("Reports: baseline_results.csv, baseline_fold_metrics.csv, baseline_config.json")

Saved: models/baseline_logistic_regression.joblib
Reports: baseline_results.csv, baseline_fold_metrics.csv, baseline_config.json


## 6. Smoke test: predict new example sentences
Apply the same preprocessing as Member 1 before inference. These invented examples demonstrate that model loading and inference work; they are not an accuracy test. A bias probability is a model estimate, not proof of bias.

In [7]:
model = joblib.load(model_path)
examples = ["The committee met on Tuesday to discuss the proposal.",
            "The committee made yet another disgraceful decision."]
for sentence in examples:
    text = preprocess_text(sentence)
    label = int(model.predict([text])[0])
    probability = model.predict_proba([text])[0, list(model.classes_).index(1)]
    print(sentence)
    print("Prediction:", "Biased" if label == 1 else "Non-biased",
          "| P(biased):", round(float(probability), 3))
    print()

The committee met on Tuesday to discuss the proposal.
Prediction: Non-biased | P(biased): 0.301

The committee made yet another disgraceful decision.
Prediction: Biased | P(biased): 0.532



## 7. Next stage (November 27)
Add Multinomial Naive Bayes, Linear SVM and Random Forest; compare them using the same training-only methodology. Hyperparameter tuning, final test metrics and the final `best_model.joblib` are later tasks.